# RazorStitch — Train DQN Recovery Policy

**Where to run:** MacBook M3 (recommended) or Google Colab.

- MacBook: ~3–8 min for 1500 episodes (CPU/MPS)
- Colab: optional for longer sweeps; GPU not required for this small MLP

Outputs:
- `eval/checkpoints/dqn_train_seed{SEED}.pt`
- `eval/checkpoints/policy_manifest.json`

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "packages").exists():
    ROOT = ROOT.parent  # running from notebooks/
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)

In [ ]:
# Colab only: uncomment to install deps
# !pip install -q -e .

In [ ]:
from packages.policy.dqn import get_device
from packages.policy.train import train_dqn

SEED = 42
EPISODES = 1500  # bump to 3000+ for final polish week

print("Device:", get_device())
agent = train_dqn(env_name="train", seed=SEED, episodes=EPISODES)
print(f"Trained {agent.steps} steps, epsilon={agent.epsilon():.4f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from packages.simulator.env import RecoveryEnv

# Quick sanity: greedy rollout on 50 episodes
env = RecoveryEnv("val", seed=SEED)
rewards = []
for ep in range(50):
    env.rng = np.random.default_rng(SEED + ep)
    env.customer.rng = env.rng
    stats = agent.run_episode(env, explore=False)
    rewards.append(stats["reward"])

plt.figure(figsize=(8, 3))
plt.plot(rewards, alpha=0.6, label="episode reward")
plt.axhline(np.mean(rewards), color="red", ls="--", label=f"mean={np.mean(rewards):.1f}")
plt.xlabel("Episode")
plt.ylabel("Net reward (INR)")
plt.title("DQN greedy validation rollouts")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
ckpt = ROOT / f"eval/checkpoints/dqn_train_seed{SEED}.pt"
print("Checkpoint:", ckpt, "exists:", ckpt.exists())

# Colab: download checkpoint
# from google.colab import files
# files.download(str(ckpt))